# Lesson 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** ఒక ఓపెన్ ప్రమాణం, ఇది ఏజెంట్లకు రన్‌টাইమ్‌లో డైనమిక్‌గా టూల్స్, వనరులు మరియు డేటా మూలాలను కనుగొని ఉపయోగించడానికి వీలు కల్పిస్తుంది. ఏజెంట్‌లో టూల్స్‌ను హార్డ్‌కోడ్ చేయడానికి బదులు, MCP ఏజెంట్లు అవసరానికి అనుగుణంగా సామర్థ్యాలను ఎక్స్‌పోజ్ చేసే బయటి సర్వర్లతో కనెక్ట్ అవడానికి అనుమతిస్తుంది.

In this lesson, you'll learn:
- MCP అంటే ఏమిటి మరియు ఇది ఏజెంట్ వ్యవస్థలకు ఎందుకు ముఖ్యం
- MCP క్లయింట్-సర్వర్ ఆర్కిటెక్చర్ ఎలా పనిచేస్తుంది
- MCP శైలిలో టూల్ కనుగొనడం ఉపయోగించే ఏజెంట్లను ఎలా నిర్మించాలో


## సెటప్

**ముందస్తు అవసరాలు:**
- అమలైన మోడల్ ఉన్న Azure AI Foundry ప్రాజెక్ట్
- ధృవీకరణ కోసం `az login` నడపండి


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మోడల్ సందర్భ ప్రోటోకాల్ (MCP) అంటే ఏమిటి?

MCP బాహ్య టూల్స్ మరియు డేటా మూలాలను కనుగొనడానికి మరియు వాటితో పరస్పర చర్యలు నిర్వహించడానికి ఒక ప్రమాణిత విధానాన్ని నిర్వచిస్తుంది:

- **MCP Server**: ఒక ప్రమాణిత ప్రోటోకాల్ ద్వారా టూల్స్, వనరులు మరియు ప్రాంప్ట్‌లను అందుబాటులోకి తెస్తుంది
- **MCP Client**: సర్వర్లతో కనెక్ట్ అయ్యే మరియు లభ్యమయ్యే సామర్థ్యాలను కనుగొనే ఏజెంట్ రన్‌టైమ్
- **Dynamic Discovery**: ఏజెంట్స్‌కు హార్డ్‌కోడెడ్ టూల్స్ అవసరం లేదు — అవి రన్‌టైమ్‌లో ఏమి అందుబాటులో ఉందో కనుగొంటాయి

ఇది విస్తరించుకునే ఏజెంట్ వ్యవస్థలను నిర్మించడంలో శక్తివంతం, ఎక్కడ కొత్త సామర్థ్యాలను ఏజెంట్ కోడ్‌ను మార్చకుండా జోడించవచ్చును.


## MCP ఎలా పనిచేస్తుంది

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ఏజెంట్ (MCP క్లయింట్) ఒక MCP సర్వర్‌కు కనెక్ట్ అవుతుంది
2. సర్వర్ లభ్యమైన సాధనాల జాబితా మరియు వాటి స్కీమాలతో స్పందిస్తుంది
3. తర్వాత ఏజెంట్ తన నిర్ణయ ప్రక్రియ సమయంలో కనుగొనబడిన ఏ సాధనాన్ని అయినా పిలవగలదు
4. ఫలితాలు అదే ప్రోటోకాల్ ద్వారా తిరిగి ప్రవహిస్తాయి


## MCP టూల్ గుర్తింపును అనుకరించడం

నిజమైన MCP సర్వర్ ఒక నడుస్తున్న సర్వర్ ప్రాసెస్ అవసరం ఉన్నందున, MCP-కి కనెక్ట్ అయిన వసతి సేవ ఏమి అందిస్తుందో అనుకరించేందుకు `@tool` ఫంక్షన్లను ఉపయోగించి మేము ఈ నమూనాను ప్రదర్శిస్తాము.

ప్రొడక్షన్‌లో, ఇవి లోకల్‌గా నిర్వచించబడటం కాకుండా MCP సర్వర్ నుండి డైనమిక్గా కనుగొనబడతాయి.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-శైలీ సాధనాలతో ఏజెంట్‌ను నిర్మించడం


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ఉత్పత్తి వాతావరణంలో

ఉత్పత్తి వాతావరణంలో, MCP శక్తివంతమైన నమూనాలను అనుమతిస్తుంది:

- **డైనామిక్ టూల్ కనుగొనడం**: ఏజెంట్లు MCP సర్వర్లతో కనెక్ట్ అవుతాయి మరియు రన్‌టైమ్‌లో టూల్‌లను కనుగొంటాయి
- **విభజింపబడిన ఆర్కిటెక్చర్**: టూల్ ప్రొవైడర్లు ఏజెంట్ నుంచి స్వతంత్రంగా అప్డేట్ చేయగలరు
- **సంస్థల మధ్య భాగస్వామ్యం**: టిమ్స్ MCP సర్వర్ల ద్వారా సామర్ధ్యాలను వెల్లడించగలవు, ఇవి ఏఏ ఏజెంట్ అయినా ఉపయోగించగలదు
- **Microsoft Agent Framework మద్దతు**: MAFకి `mcp` ఇంటిగ్రేషన్ ద్వారా అంతర్గత MCP క్లయింట్ మద్దతు ఉంది

వాస్తవ MCP సర్వర్‌ను MAFతో ఉపయోగించడానికి, మీరు `hosted_mcp_tool()` లేదా MCP క్లయింట్ ఇంటిగ్రేషన్ ద్వారా కనెక్ట్ అవుతారు.

**మరింత తెలుసుకోండి:**
- [MCP స్పెసిఫికేషన్](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework కోసం MCP మద్దతు](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నవి:
- **MCP** ఏజెంట్లు మరియు టూల్ ప్రొవైడర్ల మధ్య డైనమిక్ టూల్ కనుగొనడానికి ఓపెన్ స్టాండర్డ్
- ఈ **క్లయింట్-సర్వర్ ఆర్కిటెక్చర్** ఏజెంట్లకు రన్‌టైమ్‌లో సామర్థ్యాలను కనుగొనటానికి అనుమతిస్తుంది
- MCP **విస్తరణీయమైన, విడదీయబడిన ఏజెంట్ వ్యవస్థలను** అనుమతిస్తుంది, ఇక్కడ టూల్‌లను కోడ్ మార్పుల 없이 జోడించవచ్చు
- Microsoft Agent Framework ఉత్పత్తి వినియోగానికి **అంతర్గత MCP మద్దతు** అందిస్తుంది


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
స్పష్టీకరణ:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించారు. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా తప్పిదాలు ఉండే అవకాశముంది. అసలైన భాషలో ఉన్న మూల పత్రాన్ని అధికారిక మూలంగా పరిగణించాలి. కీలకమైన సమాచారానికి వృత్తిపరమైన మానవ అనువాదాన్ని ఉపయోగించమని సిఫార్సు చేయబడుతుంది. ఈ అనువాదాన్ని ఉపయోగించడం వల్ల ఏర్పడే ఏవైనా అపార్థాలు లేదా తప్పుగా అర్థమయ్యే పరిస్థితుల కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
